In [3]:
import pandas as pd
df = pd.read_json('Varad_data.jsonl', lines=True)
print(df.head)

<bound method NDFrame.head of                                                messages
0     [{'role': 'user', 'content': 'intro'}, {'role'...
1     [{'role': 'user', 'content': 'intro'}, {'role'...
2     [{'role': 'user', 'content': 'intro'}, {'role'...
3     [{'role': 'user', 'content': 'intro'}, {'role'...
4     [{'role': 'user', 'content': 'intro'}, {'role'...
...                                                 ...
1627  [{'role': 'user', 'content': 'Do you like talk...
1628  [{'role': 'user', 'content': 'Do you like good...
1629  [{'role': 'user', 'content': 'Do you enjoy fun...
1630  [{'role': 'user', 'content': 'Do you like inte...
1631  [{'role': 'user', 'content': 'Do you enjoy cha...

[1632 rows x 1 columns]>


In [2]:
import torch
import tiktoken
enc = tiktoken.encoding_for_model("gpt-4o")

special_tokens = {
    "<|user|>": 200100,
    "<|assistant|>": 200101,
    "<|mood|>" : 200103 ,
    "<|end|>": 200102
}
allowed = set(special_tokens.keys())

user_messege = []
asistant_reply = []
mood = []

for i in range(len(df['messages'])) :
    user_messege.append(df['messages'][i][0]['content'] )
    mood.append(df['messages'][i][1].get('mood', 'happy').lower())
    asistant_reply.append(df['messages'][i][1]['content'])

question_tokens  = []
target_tokens = []

for i in range(len(df['messages'])) :
    user_encods = enc.encode( "<|user|>"  + user_messege[i] ,  allowed_special= allowed)
    asis_encods = enc.encode( "<|mood|>" + mood[i] +  "<|assistant|>" +  asistant_reply[i] + "<|end|>"  , allowed_special= allowed )
    tokens = user_encods + asis_encods
    question = torch.tensor(tokens[:-1])
    question_tokens.append(question)
    target    = torch.tensor(tokens[1:])
    target_tokens.append(target)





In [4]:
# print(target_tokens)

In [5]:
g = torch.Generator()
g.manual_seed(42)

In [6]:
max_num = max(set(tokens))
print(max_num)

173781


In [7]:
vocab_size = len(set(tokens))
print(vocab_size)

21


In [8]:
compl_token = []
for i in range(len(question_tokens)):
    compl_token += question_tokens[i].tolist()
    compl_token += target_tokens[i].tolist()

vocab = sorted(set(compl_token))
stoi = {s:i for i,s in enumerate(vocab)}
itos = {i:s for i,s in enumerate(vocab)}
vocab_size = len(vocab)
q_tokens = []
t_tokens = []
for i in range(len(question_tokens)):
    q_tensor = torch.tensor([stoi[tok.item()] for tok in question_tokens[i]])
    t_tensor = torch.tensor([stoi[tok.item()] for tok in target_tokens[i]])
    q_tokens.append(q_tensor)
    t_tokens.append(t_tensor)

In [9]:
q_tokens = torch.cat(q_tokens)
t_tokens = torch.cat(t_tokens)

In [10]:
block_size = 64
xs = []
ys = []

for i in range(len(q_tokens)):
    context = q_tokens[max(0, i-block_size+1) : i+1]
    context = torch.nn.functional.pad(context, (block_size - len(context), 0))
    xs.append(context)
    ys.append(t_tokens[i])

xs = torch.stack(xs)
ys = torch.stack(ys)

In [11]:
dim = 300

# Encods  = torch.randn((vocab_size, dim), requires_grad=True)
# W1 = torch.randn((64*dim, dim), requires_grad=True)
# W2 = torch.randn((dim, vocab_size), requires_grad=True)
# enc = Encods[xs]
# update_enc = enc.view(enc.shape[0] , -1 )
# hidden = torch.tanh(update_enc @ W1 )
# logits = hidden @ W2

In [12]:
# b_gain = torch.ones((1, 300), requires_grad=True)
# b_bias = torch.zeros((1, 300), requires_grad=True)
# parameters  = { Encods , W1 , W2  , b_gain , b_bias}

In [13]:
# logits.shape


In [14]:
# loss = torch.nn.functional.cross_entropy(logits, ys)
# print(loss)

In [15]:
# from google.colab import files
# uploaded = files.upload()

# checkpoint = torch.load('Sakhi.pt')
# Encods      = checkpoint['Encods']
# W1     = checkpoint['W1']
# W2     = checkpoint['W2']
# b_gain = checkpoint['b_gain']
# b_bias = checkpoint['b_bias']
# stoi   = checkpoint['stoi']
# itos   = checkpoint['itos']

# vocab_size = len(stoi)
# parameters  = { Encods , W1 , W2  , b_gain , b_bias}

In [16]:
# batch_size = 32
# prev_loss = 100000
# for i in range(1):
#     ix = torch.randint(0, xs.shape[0], (batch_size,))
#     enc  = Encods[xs[ix]]
#     update_enc = enc.view(enc.shape[0], -1)
#     flow_layer = update_enc @ W1
#     flow_layer = b_gain * (flow_layer - flow_layer.mean(0)) / flow_layer.std(0) + b_bias
#     hidden     = torch.tanh(flow_layer)
#     logits     = hidden @ W2
#     loss       = torch.nn.functional.cross_entropy(logits, ys[ix])

#     for p in parameters:
#         p.grad = None
#     loss.backward()
#     lr = 0.001
#     for p in parameters:
#         p.data += -lr * p.grad

#     if prev_loss < loss:
#         prev_loss = loss
#         break
# print(loss)

In [17]:
# torch.save({'Encods' : Encods,'W1'     : W1,'W2'     : W2,'b_gain' : b_gain,    'b_bias' : b_bias,'stoi'   : stoi,'itos'   : itos,}, 'Amika.pt')
# from google.colab import files
# files.download('Sakhi.pt')


In [18]:
import torch.nn as nn

In [19]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
positions  = nn.Embedding(block_size, 300).to(device)

In [20]:
print(device)

cpu


In [21]:
tokenizer = tiktoken.encoding_for_model('gpt-4o')
size = max(len(tokenizer.encode('<|user|>' + user_messege[i] + '<|assistant|>' + asistant_reply[i] + '<|end|>', allowed_special=allowed)) for i in range(len(df)))
print(size)

298


In [22]:
# head_quant = 25
# mask = torch.tril(torch.ones(size, size)).to(device)
# query  = nn.Linear(dim , head_quant , bias = False )
# key  = nn.Linear(dim , head_quant , bias = False )
# value  = nn.Linear(dim , head_quant , bias = False )
# q = query(x)
# k = key(x)
# v = value(x)
# wieghts  = q @ k.transpose(-2,-1) * ((head_quant) ** ( -0.5 ))
# wieghts = wieghts.masked_fill(mask[:block_size, :block_size] == 0, float('-inf'))
# wieghts = torch.softmax(wieghts , dim = -1 )
# pos_pattern = wieghts @ v

In [39]:
dim        = 300
n_layers   = 4
num_heads  = 4
head_quant = 64

In [40]:
head_quant = 75
class Head(nn.Module):
    def __init__(self, head_quant):
        super().__init__()
        self.query = nn.Linear(dim, head_quant, bias=False)
        self.key   = nn.Linear(dim, head_quant, bias=False)
        self.value = nn.Linear(dim, head_quant, bias=False)
        self.mask  = torch.tril(torch.ones(block_size, block_size))

    def forward(self, x):
        q = self.query(x)
        k = self.key(x)
        v = self.value(x)
        T = x.shape[1]
        wieghts = q @ k.transpose(-2,-1) * (head_quant ** -0.5)
        wieghts = wieghts.masked_fill(self.mask[:T, :T].to(x.device) == 0, float('-inf'))
        wieghts = torch.softmax(wieghts, dim=-1)
        pos_pattern = wieghts @ v
        return pos_pattern

In [41]:
class head_concat(nn.Module):
    def __init__(self, num_heads, head_quant):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_quant) for _ in range(num_heads)])
        self.proj  = nn.Linear(dim, dim)

    def forward(self, x):
        pos_pattern = torch.cat([head(x) for head in self.heads], dim=-1)
        return self.proj(pos_pattern)

In [42]:
class block_in_transformer(nn.Module) :

    def __init__(self) :
         super().__init__()
         self.head = head_concat(num_heads, head_quant)
         self.hidden_layer_1 = nn.Linear(dim , 4*dim).to(device)
         self.hidden_layer_2  = nn.Linear(4*dim , dim).to(device)
         self.norm_1 = nn.LayerNorm(dim)
         self.norm_2 = nn.LayerNorm(dim)


    def forward(self , x) :
       pattern = self.head(x)
       x = x + pattern
       x  = self.norm_1(x)
       ffn = torch.nn.functional.leaky_relu(self.hidden_layer_1(x), negative_slope=0.01)
       ffn = self.hidden_layer_2(ffn)
       x = x + ffn
       x = self.norm_2(x)
       return x




In [43]:
Blocks = nn.ModuleList([block_in_transformer() for _ in range(n_layers)]).to(device)
Encods  = nn.Embedding(vocab_size, dim).to(device)
final = nn.Linear( dim , vocab_size).to(device)
parameters = list(Encods.parameters()) + list(positions.parameters()) + list(Blocks.parameters()) + list(final.parameters())


In [44]:
optimizer = torch.optim.AdamW(parameters, lr=3e-4)

In [45]:
block_size = 64
xs = []
ys = []

for i in range(0, len(q_tokens) - block_size):
    xs.append(q_tokens[i : i + block_size])
    ys.append(t_tokens[i : i + block_size])

xs = torch.stack(xs)
ys = torch.stack(ys)

In [30]:
# from google.colab import files
# uploaded = files.upload()

# checkpoint = torch.load('Sakhi_transformer.pt')
# Encods.load_state_dict(checkpoint['Encods'])
# positions.load_state_dict(checkpoint['positions'])
# Blocks.load_state_dict(checkpoint['blocks'])
# final.load_state_dict(checkpoint['final'])
# stoi = checkpoint['stoi']
# itos = checkpoint['itos']

# vocab_size = len(stoi)


In [ ]:
batch_size = 128
for i in range(100):
    ix  = torch.randint(0, xs.shape[0], (batch_size,))
    x_  = xs[ix].to(device)
    y_  = ys[ix].to(device)

    tok_emb = Encods(x_)
    pos_emb = positions(torch.arange(block_size).to(device))
    x = tok_emb + pos_emb
    for block in Blocks:
        x = block(x)

    logits = final(x)
    loss   = torch.nn.functional.cross_entropy(logits.view(-1, vocab_size),y_.view(-1))

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    print(loss)

In [ ]:
torch.save({
    'Encods'  : Encods.state_dict(),
    'positions'  : positions.state_dict(),
    'blocks'     : Blocks.state_dict(),
    'final'      : final.state_dict(),
    'stoi'       : stoi,
    'itos'       : itos,
}, 'Sakhi.pt')

from google.colab import files
files.download('Sakhi.pt')

In [ ]:
def generate(prompt, max_tokens=50):
    tokenizer = tiktoken.encoding_for_model('gpt-4o')
    tokens = tokenizer.encode('<|user|>' + prompt, allowed_special=allowed)
    tokens = [stoi[t] for t in tokens if t in stoi]
    x = torch.tensor(tokens).unsqueeze(0).to(device)

    for _ in range(max_tokens):
        x_crop = x[:, -block_size:]
        tok_emb = Encods(x_crop)
        pos_emb = positions(torch.arange(x_crop.shape[1]).to(device))
        out = tok_emb + pos_emb
        for block in Blocks:
            out = block(out)
        logits = final(out)
        next_token = logits[:, -1, :].argmax(dim=-1)
        x = torch.cat([x, next_token.unsqueeze(0)], dim=1)

    return tokenizer.decode([itos[t] for t in x[0].tolist()])

print(generate('Tell about his  projects '))